# Multi-class regression

In this experiment we will use MNIST dataset to build a multi-class regression model.

In [122]:
from datasets import load_dataset

from tqdm import tqdm
import random

import torch as t
from torch.nn import Module, Linear, ReLU, Softmax, CrossEntropyLoss
from torch.optim import AdamW
from torch.utils.data import DataLoader

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from torchvision.transforms import v2

device = "cpu"
if t.cuda.is_available():
    device = "cuda"
elif t.mps.is_available():
    device = "mps"

# Site color palette (matches the blog theme)
SITE = dict(
    bg_primary   = '#213636',
    bg_secondary = '#1a2d2d',
    bg_tertiary  = '#2a4444',
    border       = '#3a5454',
    text_primary = '#c8c4b8',
    text_secondary = '#a09890',
    accent       = '#bdb76b',
    olive        = '#828631',
)

t.manual_seed(42)

## Load the dataset using hugging face datasets library

Datasource: https://huggingface.co/datasets/ylecun/mnist

In [123]:
ds = load_dataset("ylecun/mnist")
mean, std = 0.1307, 0.3081

In [124]:
IMG_TX = v2.Compose([
    v2.PILToTensor(),
    v2.ToDtype(dtype=t.float32, scale=True),    # scale=True: uint8 [0,255] -> float [0,1]
    v2.Normalize((mean,), (std,))               # normalise
])

def trainsform_data(batch):
    batch["image"] = [IMG_TX(img).flatten() for img in batch["image"]]

    return batch

ds = ds.with_transform(trainsform_data)

print(f"Training samples: {ds["train"].shape}")
print(f"Testing samples: {ds["test"].shape}")
print(f"Image shape: {ds["train"][0]["image"].shape}")

Training samples: (60000, 2)
Testing samples: (10000, 2)
Image shape: torch.Size([784])


In [125]:
class DigitClassifier(Module):

    def __init__(self, in_feat=784, out_feat=10):
        super().__init__()
        self.linear1 = Linear(in_features=in_feat, out_features=64)
        self.relu = ReLU()
        self.linear2 = Linear(in_features=64, out_features=out_feat)

        t.nn.init.kaiming_normal_(self.linear1.weight)
        t.nn.init.kaiming_normal_(self.linear2.weight)

    def forward(self, x: t.Tensor) -> t.Tensor:
        h = self.linear1(x)
        h = self.relu(h)
        return self.linear2(h)


def test_model():
    expected = (10,)
    x = t.rand(784, dtype=t.float32)
    model = DigitClassifier()
    out = model(x)
    assert out.shape == expected, f"Shapes don't match. expected={expected} actual={out.shape}"

test_model()

In [126]:
generator = t.Generator(device="cpu")
train_data = DataLoader(
    ds["train"],
    batch_size=512,
    shuffle=True,
    num_workers=0,
    generator=generator,
)
test_data = DataLoader(
    ds["test"],
    batch_size=512,
    num_workers=0,
    generator=generator,
)

In [127]:
def accuracy(output: t.Tensor, target: t.Tensor):
    return (output == target).type(t.float32).mean()

def train(
    model: Module, 
    train_data: DataLoader, 
    test_data: DataLoader,
    ce_loss=CrossEntropyLoss(),
    epochs=50, 
    lr=1e-3,
):

    optim = AdamW(model.parameters(), lr=lr)
    epoch_losses, epoch_accuracies = [], []
    for epoch in tqdm(range(epochs), desc="Epochs"):
        losses = []
        model.train()
        for batch in train_data:
            input = batch["image"].to(device)
            target = batch["label"].to(device)
            output = model(input)
            loss = ce_loss(output, target)

            losses.append(loss.detach())

            optim.zero_grad()
            loss.backward()
            optim.step()

        batch_loss = t.stack(losses).mean()
        epoch_losses.append(batch_loss.item())

        model.eval()
        with t.no_grad():
            accs = []
            for batch in test_data:
                input = batch["image"].to(device)
                target = batch["label"].to(device)
                output = model(input)
                output = output.softmax(dim=-1)
                output = output.argmax(dim=-1)

                acc = accuracy(output, target)
                accs.append(acc)

        batch_acc = t.stack(accs).mean()
        epoch_accuracies.append(batch_acc.item())

        if (epoch+1) % 5 == 0:
            print(f"Epoch: {epoch+1}, Loss: {batch_loss:.4f} Accuracy: {batch_acc:.4f}")

    return model, epoch_losses, epoch_accuracies


model = DigitClassifier().to(device)
model, losses, accuracies = train(model=model, train_data=train_data, test_data=test_data)

Epochs:  10%|█         | 5/50 [00:16<02:27,  3.27s/it]

Epoch: 5, Loss: 0.1247 Accuracy: 0.9590


Epochs:  20%|██        | 10/50 [00:32<02:11,  3.29s/it]

Epoch: 10, Loss: 0.0696 Accuracy: 0.9698


Epochs:  30%|███       | 15/50 [00:49<01:54,  3.27s/it]

...

Epoch: 40, Loss: 0.0062 Accuracy: 0.9751


Epochs:  90%|█████████ | 45/50 [02:28<00:16,  3.32s/it]

Epoch: 45, Loss: 0.0048 Accuracy: 0.9753


Epochs: 100%|██████████| 50/50 [02:45<00:00,  3.30s/it]

Epoch: 50, Loss: 0.0034 Accuracy: 0.9744


In [131]:
def plot_curves(losses, accuracies):
    x = list(range(len(losses)))
    fig = make_subplots(rows=2, cols=1)
    fig.add_trace(
        go.Scatter(x=x, y=losses, mode="lines", name="Loss",
                   line=dict(color=SITE['accent'])),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=x, y=accuracies, mode="lines", name="Accuracy",
                   line=dict(color=SITE['olive'])),
        row=2, col=1,
    )
    fig.update_layout(
        title=dict(text="Loss vs accuracy plot",
                    font=dict(color=SITE['text_primary'])),
        paper_bgcolor=SITE['bg_secondary'],
        plot_bgcolor=SITE['bg_primary'],
        font=dict(color=SITE['text_primary']),
        legend=dict(orientation="h", x=0.5, y=-0.2, xanchor="center", yanchor="top",
                    bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=60),
    )
    fig.update_xaxes(title="Epochs", color=SITE['text_secondary'], gridcolor=SITE['border'])
    fig.update_yaxes(title="Value", color=SITE['text_secondary'], gridcolor=SITE['border'])
    fig.show()

plot_curves(losses, accuracies)

In [136]:
def show_image(img: t.Tensor, target: int):
    img = img * std + mean

    fig = px.imshow(img, color_continuous_scale="gray_r")
    fig.update_layout(
        title=dict(text=f"Prediction: {target}", font=dict(color=SITE['text_primary'])),
        coloraxis_showscale=False,
        paper_bgcolor=SITE['bg_secondary'],
        font=dict(color=SITE['text_primary']),
        margin=dict(l=20, r=20, t=40, b=20),
    )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor="x")
    fig.show()

def validate_model():
    idx = random.randint(0, len(ds["test"]["image"]))
    img = ds["test"]["image"][idx]
    
    out = model(img.to(device))
    out = out.softmax(dim=-1)
    target = out.argmax(dim=-1)

    show_image(img.detach().cpu().reshape(28, 28), target)

validate_model()